# Part 1: Data Preparation
## 1. Dataset Loading
### Split into training and testing subsets

In [21]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib as plt
import numpy as np
from tensorflow.keras import layers, Model

In [22]:
import tensorflow as tf

# Detect TPU
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print('Running on TPU:', tpu.master())
except ValueError:
    tpu = None

if tpu:
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print('TPU detected')
else:
    strategy = tf.distribute.get_strategy()
    print('Running on CPU/GPU')

print(f'Number of replicas: {strategy.num_replicas_in_sync}')

Running on CPU/GPU
Number of replicas: 1


In [23]:
def build_generator(latent_dim=100):
    """Builds the generator model for 64x64 RGB images"""
    
    model = tf.keras.Sequential([
        # Start with latent vector (100-dim)
        tf.keras.layers.Dense(8*8*256, use_bias=False, input_shape=(latent_dim,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        
        # Reshape to 8x8x256
        tf.keras.layers.Reshape((8, 8, 256)),
        
        # Upsample to 16x16
        tf.keras.layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        
        # Upsample to 32x32
        tf.keras.layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        
        # Upsample to 64x64
        tf.keras.layers.Conv2DTranspose(32, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        
        # Output layer (64x64x3)
        tf.keras.layers.Conv2DTranspose(3, (5, 5), strides=(1, 1), padding='same', use_bias=False,
                                       activation='tanh')  # Output in [-1, 1] range
        
    ])
    
    return model

# Test generator
generator = build_generator()
noise = tf.random.normal([1, 100])
generated_image = generator(noise, training=False)
print(f"Generator output shape: {generated_image.shape}")

Generator output shape: (1, 64, 64, 3)


In [24]:
def build_discriminator():
    """Builds the discriminator model for 64x64 RGB images"""
    
    model = tf.keras.Sequential([
        # Input: 64x64x3
        tf.keras.layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same',
                              input_shape=[64, 64, 3]),
        tf.keras.layers.LeakyReLU(alpha=0.2),
        tf.keras.layers.Dropout(0.3),
        
        # Downsample to 32x32
        tf.keras.layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        tf.keras.layers.LeakyReLU(alpha=0.2),
        tf.keras.layers.Dropout(0.3),
        
        # Downsample to 16x16
        tf.keras.layers.Conv2D(256, (5, 5), strides=(2, 2), padding='same'),
        tf.keras.layers.LeakyReLU(alpha=0.2),
        tf.keras.layers.Dropout(0.3),
        
        # Flatten and output
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1, activation='sigmoid')  # Binary classification: real/fake
    ])
    
    return model

# Test discriminator
discriminator = build_discriminator()
decision = discriminator(generated_image)
print(f"Discriminator output: {decision}")

Discriminator output: [[0.5]]


In [25]:
# Initialize TPU strategy
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print('Running on TPU:', tpu.master())
except ValueError:
    strategy = tf.distribute.get_strategy()
    print('Running on CPU/GPU')

print(f'Number of replicas: {strategy.num_replicas_in_sync}')

# Hyperparameters optimized for TPU
BATCH_SIZE = 128 * strategy.num_replicas_in_sync  # Scale with TPU cores
LATENT_DIM = 100
EPOCHS = 200

# Mixed precision for TPU
tf.keras.mixed_precision.set_global_policy('mixed_bfloat16')

# Create models within TPU strategy
with strategy.scope():
    # Build generator and discriminator
    generator = build_generator(LATENT_DIM)
    discriminator = build_discriminator()
    
    # Optimizers
    generator_optimizer = tf.keras.optimizers.Adam(
        learning_rate=0.0002, 
        beta_1=0.5,  # Common for GANs
        clipnorm=1.0  # Gradient clipping for stability
    )
    
    discriminator_optimizer = tf.keras.optimizers.Adam(
        learning_rate=0.0002,
        beta_1=0.5,
        clipnorm=1.0
    )
    
    # Loss function
    cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=False)
    
    # Define loss functions
    def discriminator_loss(real_output, fake_output):
        real_loss = cross_entropy(tf.ones_like(real_output), real_output)
        fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
        total_loss = real_loss + fake_loss
        return total_loss
    
    def generator_loss(fake_output):
        return cross_entropy(tf.ones_like(fake_output), fake_output)

Running on CPU/GPU
Number of replicas: 1


In [ ]:
def create_gan_dataset(batch_size=128):
    """Create TPU-optimized dataset for GAN training"""
    
    # Load dataset (only need images, not labels for GAN)
    (ds_train, _), ds_info = tfds.load(
        'cats_vs_dogs',
        split=['train[:90%]', 'train[90%:]'],
        shuffle_files=True,
        as_supervised=True,
        with_info=True,
    )
    
    def preprocess(image, label):
        # Resize
        image = tf.image.resize(image, [64, 64])
        # Normalize to [-1, 1] for GAN
        image = (tf.cast(image, tf.float32) / 127.5) - 1
        return image, label  # Return dummy label for compatibility
    
    # Apply preprocessing
    ds_train = ds_train.map(
        preprocess,
        num_parallel_calls=tf.data.experimental.AUTOTUNE
    )
    
    # Optimize dataset for TPU
    ds_train = ds_train.shuffle(buffer_size=1000)
    ds_train = ds_train.batch(batch_size, drop_remainder=True)  # Important for TPU!
    ds_train = ds_train.prefetch(tf.data.experimental.AUTOTUNE)
    
    return ds_train

# Create dataset
dataset = create_gan_dataset(BATCH_SIZE)
print(f"Batch size: {BATCH_SIZE}")
print(f"Dataset element spec: {dataset.element_spec}")

In [ ]:
def preprocess(image, label):
    # Resize images
    image = tf.image.resize(image, [64, 64])
    
    # Normalize to [-1, 1] range
    image = (tf.cast(image, tf.float32) / 127.5) - 1
    
    # Cast labels
    label = tf.cast(label, tf.int32)
    
    return image, label

def configure_for_performance(ds, batch_size=128):
    # Use buffered prefetching for better performance
    ds = ds.cache()
    ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(batch_size, drop_remainder=True)  # Important for TPU
    ds = ds.prefetch(tf.data.experimental.AUTOTUNE)
    return ds

# Apply preprocessing
ds_train = ds_train.map(
    preprocess, 
    num_parallel_calls=tf.data.experimental.AUTOTUNE
)

# Configure for performance
batch_size = 128 * strategy.num_replicas_in_sync  # Scale batch size with TPU cores
ds_train = configure_for_performance(ds_train, batch_size)

## 2. Preprocessing:
### o Resize all images to 64×64 pixels
### o Normalize pixel values to the [-1, 1] range
### o Cast labels to integer format
### o Create training batches with labels

# Part 2: Conditional GAN Architecture
## 1. Generator
### o Accepts both noise vector and class label as input
### o Uses label embedding and concatenates it with noise
### o Builds up to a 64×64×3 image using Conv2DTranspose layers

## 2. Discriminator
### o Accepts both real/fake image and class label
### o Embeds label and concatenates it with image
### o Uses Conv2D layers to classify input as real/fake

## Part 3: Loss Functions and Optimizer
### • Generator Loss: Binary cross-entropy encouraging discriminator to classify generated images as real
### • Discriminator Loss: Binary cross-entropy distinguishing real vs fake images
### • Optimizers: Adam optimizer for both networks


## Part 4: Training the Model
### • Train the generator and discriminator alternately
### • Use @tf.function for optimized performance
### • Train for a minimum of 10 epochs
### • Print training progress per epoch


In [ ]:
# Training step with TPU optimization
@tf.function
def train_step(images):
    # Generate random noise
    batch_size = tf.shape(images)[0]
    noise = tf.random.normal([batch_size, LATENT_DIM])
    
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # Generate images
        generated_images = generator(noise, training=True)
        
        # Discriminator decisions
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)
        
        # Calculate losses
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)
    
    # Calculate gradients
    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    
    # Apply gradients
    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))
    
    return gen_loss, disc_loss

## 5. Complete GAN Training Function for 200 Epochs

In [ ]:
def train_gan(dataset, epochs=200):
    """Train GAN for specified number of epochs with TPU optimization"""
    
    # Metrics tracking
    gen_loss_metric = tf.keras.metrics.Mean(name='gen_loss')
    disc_loss_metric = tf.keras.metrics.Mean(name='disc_loss')
    
    # History
    history = {
        'gen_loss': [],
        'disc_loss': [],
        'epoch_time': []
    }
    
    # Generate fixed noise for visualization
    fixed_noise = tf.random.normal([16, LATENT_DIM])
    
    for epoch in range(epochs):
        start_time = time.time()
        
        # Reset metrics
        gen_loss_metric.reset_states()
        disc_loss_metric.reset_states()
        
        # Train for one epoch
        for batch in dataset:
            images, _ = batch
            gen_loss, disc_loss = train_step(images)
            
            # Update metrics
            gen_loss_metric.update_state(gen_loss)
            disc_loss_metric.update_state(disc_loss)
        
        # Calculate epoch time
        epoch_time = time.time() - start_time
        
        # Get average losses
        avg_gen_loss = gen_loss_metric.result().numpy()
        avg_disc_loss = disc_loss_metric.result().numpy()
        
        # Save to history
        history['gen_loss'].append(avg_gen_loss)
        history['disc_loss'].append(avg_disc_loss)
        history['epoch_time'].append(epoch_time)
        
        # Print progress
        print(f'Epoch {epoch+1}/{epochs}, '
              f'Gen Loss: {avg_gen_loss:.4f}, '
              f'Disc Loss: {avg_disc_loss:.4f}, '
              f'Time: {epoch_time:.2f}s')
        
        # Generate sample images every 10 epochs
        if (epoch + 1) % 10 == 0:
            generate_and_save_images(generator, epoch + 1, fixed_noise)
            
            # Save model checkpoints
            if (epoch + 1) % 50 == 0:
                save_model_checkpoints(generator, discriminator, epoch + 1)
        
        # Learning rate decay
        if (epoch + 1) % 50 == 0:
            decay_learning_rate(generator_optimizer, discriminator_optimizer)
        
        # Clear memory every 20 epochs
        if (epoch + 1) % 20 == 0:
            tf.keras.backend.clear_session()
            gc.collect()
    
    return history

def generate_and_save_images(model, epoch, test_input):
    """Generate and save images for visualization"""
    predictions = model(test_input, training=False)
    
    fig = plt.figure(figsize=(4, 4))
    
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        # Convert from [-1, 1] to [0, 1] for display
        img = (predictions[i] + 1) / 2.0
        plt.imshow(img)
        plt.axis('off')
    
    plt.suptitle(f'Epoch {epoch}', fontsize=14)
    plt.savefig(f'image_at_epoch_{epoch:04d}.png')
    plt.close()

def save_model_checkpoints(generator, discriminator, epoch):
    """Save model checkpoints"""
    generator.save(f'generator_epoch_{epoch}.h5')
    discriminator.save(f'discriminator_epoch_{epoch}.h5')
    print(f'Models saved at epoch {epoch}')

def decay_learning_rate(g_optimizer, d_optimizer, decay_factor=0.5):
    """Decay learning rate"""
    g_optimizer.learning_rate.assign(g_optimizer.learning_rate * decay_factor)
    d_optimizer.learning_rate.assign(d_optimizer.learning_rate * decay_factor)
    print(f'Learning rate decayed to {g_optimizer.learning_rate.numpy()}')

## 6. Data Pipeline Optimization for GAN

## 7. Start Training

In [ ]:
# Start GAN training
print("Starting GAN training...")
print(f"Training for {EPOCHS} epochs with batch size {BATCH_SIZE}")

# Train the GAN
history = train_gan(dataset, epochs=EPOCHS)

# Plot training history
def plot_training_history(history):
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 3, 1)
    plt.plot(history['gen_loss'], label='Generator Loss')
    plt.plot(history['disc_loss'], label='Discriminator Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('GAN Training Loss')
    
    plt.subplot(1, 3, 2)
    plt.plot(history['epoch_time'])
    plt.xlabel('Epoch')
    plt.ylabel('Time (s)')
    plt.title('Epoch Time')
    
    plt.subplot(1, 3, 3)
    # Plot loss ratio
    loss_ratio = [g/d for g, d in zip(history['gen_loss'], history['disc_loss'])]
    plt.plot(loss_ratio)
    plt.xlabel('Epoch')
    plt.ylabel('Gen Loss / Disc Loss')
    plt.title('Loss Ratio')
    plt.axhline(y=1, color='r', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig('gan_training_history.png')
    plt.show()

plot_training_history(history)

## Part 5: Evaluation and Visualization
### • Generate images conditioned on randomly chosen labels
### • Display generated images with corresponding label titles
### • Ensure images are normalized back to [0, 1] for display
